# Fake News Detection Model Testing

This notebook demonstrates how to use the trained fake news detection model for inference.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch
from inference import FakeNewsPredictor, load_model, predict_from_dataframe
import config


## Load Trained Model

Load a trained model checkpoint. If no checkpoint exists, the model will use pre-trained weights only.


In [ ]:
# Check if we have a trained model
model_path = config.CHECKPOINT_DIR / "best_model.pt"
if model_path.exists():
    print(f"Loading trained model from {model_path}")
    predictor = load_model(model_path)
else:
    print("No trained model found. Using pre-trained weights only.")
    print("Train the model first using: python train.py")
    predictor = FakeNewsPredictor(use_retrieval=True)


## Single Post Prediction

Test the model on individual posts.


In [ ]:
# Test on a single post
test_post = "Compartilhem essa notícia urgente sobre política e saúde pública!"

result = predictor.predict(test_post)

print(f"Post: {test_post}")
print(f"\nPrediction: {'FAKE NEWS' if result['is_fake'] else 'NOT FAKE'}")
print(f"Fake Probability: {result['fake_probability']:.4f}")
print(f"Not Fake Probability: {result['not_fake_probability']:.4f}")
print(f"\nProbabilities: {result['probabilities']}")


## Batch Prediction

Predict on multiple posts at once.


In [ ]:
# Test on multiple posts
test_posts = [
    "Notícia importante sobre saúde pública e vacinação",
    "Mais uma fake news sendo desmentida pelos fatos",
    "Informação verificada e confirmada pelos especialistas",
    "Compartilhem urgentemente essa revelação chocante!"
]

results = predictor.predict_batch(test_posts)

for post, result in zip(test_posts, results):
    print(f"\nPost: {post}")
    print(f"  Prediction: {'FAKE NEWS' if result['is_fake'] else 'NOT FAKE'}")
    print(f"  Fake Probability: {result['fake_probability']:.4f}")


## Prediction with Aggregation

Use multiple claims and aggregate the results for more robust predictions.


In [ ]:
# Test with aggregation (uses top-k claims)
test_post_agg = "Informação sobre política e eleições que precisa ser compartilhada!"

result_agg = predictor.predict_with_aggregation(
    test_post_agg,
    top_k=3,
    aggregation_method="max"
)

print(f"Post: {test_post_agg}")
print(f"\nPrediction: {'FAKE NEWS' if result_agg['is_fake'] else 'NOT FAKE'}")
print(f"Fake Probability: {result_agg['fake_probability']:.4f}")
print(f"Number of claims used: {result_agg['num_claims_used']}")
print(f"Aggregation method: {result_agg['aggregation_method']}")


## Predict on Dataset

Load and predict on the annotated dataset to evaluate performance.


In [ ]:
# Load annotated dataset
df_annotated = pd.read_excel(config.ANNOTATED_DATASET_PATH)
df_annotated = df_annotated[df_annotated["is_fn"].isin([0.0, 1.0])].copy()

print(f"Total posts: {len(df_annotated)}")
print(f"Fake news: {(df_annotated['is_fn'] == 1).sum()}")
print(f"Not fake: {(df_annotated['is_fn'] == 0).sum()}")

# Sample a subset for testing (to avoid long processing times)
df_sample = df_annotated.sample(n=min(100, len(df_annotated)), random_state=42)
print(f"\nTesting on {len(df_sample)} posts...")


In [ ]:
# Predict on the sample
df_predictions = predict_from_dataframe(
    df_sample,
    post_column="message",
    claim_column="resumo_2",
    model_path=model_path if model_path.exists() else None,
    use_retrieval=False  # Use existing claims from dataset
)

# Calculate metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

y_true = df_predictions["is_fn"].astype(int)
y_pred = df_predictions["prediction"]

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print(f"\nMetrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"\nConfusion Matrix:")
print(cm)


## Analyze Predictions

Examine specific examples of correct and incorrect predictions.


In [ ]:
# Show examples of fake news predictions
print("Examples of predicted FAKE NEWS:")
print("=" * 100)
fake_examples = df_predictions[df_predictions["prediction"] == 1].head(5)
for idx, row in fake_examples.iterrows():
    print(f"\nPost: {row['message'][:200]}...")
    print(f"True Label: {'FAKE' if row['is_fn'] == 1 else 'NOT FAKE'}")
    print(f"Predicted Probability: {row['fake_probability']:.4f}")
    print(f"Match: {'✓' if row['prediction'] == row['is_fn'] else '✗'}")
    print("-" * 100)


## Save Results

Save predictions to a file for further analysis.


In [ ]:
# Save predictions
output_path = "predictions_results.xlsx"
df_predictions[["message", "is_fn", "prediction", "fake_probability", "resumo_2"]].to_excel(
    output_path,
    index=False
)
print(f"Predictions saved to {output_path}")
